# [Sensitivity Analysis] The Smoothness Constraint (KAN-TabNet) | Step LR | Higgs Boson

**Optimized Reference Configuration:** `n_d=9, n_a=9`, `kan_grid_size=3`, `kan_spline_order=3`, `initial_lr=0.0025`

### Methodological Context: The Control Variables
To precisely isolate the effects of mathematical curvature, this sensitivity analysis uses the exact same `9x9` internal routing capacity, constrained grid resolution (`G=3`), and thermodynamic optimization environment (StepLR schedule, `initial_lr=0.0025`) as our optimized reference configuration.

By fixing the structural dimensions and the learning rate, we strictly guarantee that any performance variations observed in this notebook are purely the result of altering the mathematical geometry of the Kolmogorov-Arnold Network's underlying B-splines.

### The Hypothesis
In this study, we evaluate a restricted mathematical geometry by reducing the spline order from cubic to quadratic (`kan_spline_order=2`).

The physical nature of the Higgs Boson dataset relies heavily on continuous kinematic variables and smooth invariant mass distributions. Cubic splines ($k=3$) are mathematically predisposed to mapping continuous, wave-like, and smooth functions. Quadratic splines ($k=2$), by contrast, are mathematically "sharper" and more rigid. We hypothesize that continuous quantum physics data possesses a strict "Smoothness Constraint," meaning the rigid $k=2$ splines will struggle to accurately map the smooth physical curves of the signal distributions. This notebook serves to empirically evaluate whether dropping the spline order introduces a purely mathematical bottleneck, validating the necessity of $k=3$ for continuous physics datasets.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import pandas as pd
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import gc

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz'
df = pd.read_csv(url, header=None)

# Column 0 is the class label (1 for signal, 0 for background). Columns 1-28 are features.
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values.astype(int)

# Free up the massive raw DataFrame from memory
del df
gc.collect()

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"\nFinal Train shape: {X_train.shape}")
print(f"Final Valid shape: {X_valid.shape}")
print(f"Final Test shape:  {X_test.shape}\n")


Final Train shape: (8800000, 28)
Final Valid shape: (1100000, 28)
Final Test shape:  (1100000, 28)



### Model Training

In [5]:
MAX_EPOCHS = 500

In [6]:
# Hyperparameters from original paper (TabNet-S model).
# Notes:
# - momentum is 1 - 0.6 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 37 epochs approximates the paper's 20k iterations
#   (based on a batch size of 16384 and ~8.8M training samples).
paper_config = {
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-6,
    'momentum': 0.4,
    'optimizer_fn': torch.optim.Adam,
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=37, gamma=0.9),
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=9,
    n_a=9,
    **paper_config,
    clip_value=2.,
    optimizer_params=dict(lr=0.0025),
    use_kan=True,
    kan_grid_size=3,
    kan_spline_order=2,
    seed=seed,
    verbose=25
)

In [7]:
# Hyperparameters from original paper (TabNet-S model).
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=75,
    num_workers=os.cpu_count(),
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 0.63251 | valid_accuracy: 0.66755 |  0:01:46s
epoch 25 | loss: 0.515   | valid_accuracy: 0.74252 |  0:45:40s
epoch 50 | loss: 0.49078 | valid_accuracy: 0.7572  |  1:29:36s
epoch 75 | loss: 0.48613 | valid_accuracy: 0.76032 |  2:13:25s
epoch 100| loss: 0.48127 | valid_accuracy: 0.76283 |  2:57:23s
epoch 125| loss: 0.47973 | valid_accuracy: 0.76368 |  3:41:20s
epoch 150| loss: 0.47872 | valid_accuracy: 0.76381 |  4:25:09s
epoch 175| loss: 0.47799 | valid_accuracy: 0.76437 |  5:09:07s
epoch 200| loss: 0.47687 | valid_accuracy: 0.76552 |  5:53:07s
epoch 225| loss: 0.47598 | valid_accuracy: 0.76591 |  6:37:07s
epoch 250| loss: 0.47661 | valid_accuracy: 0.76568 |  7:21:08s
epoch 275| loss: 0.47537 | valid_accuracy: 0.76546 |  8:05:01s
epoch 300| loss: 0.47414 | valid_accuracy: 0.7669  |  8:49:00s
epoch 325| loss: 0.47422 | valid_accuracy: 0.76717 |  9:33:10s
epoch 350| loss: 0.47399 | valid_accuracy: 0.76738 |  10:17:28s
epoch 375| loss: 0.47319 | valid_accuracy: 0.76773 |  

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 69,422


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.76785


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/higgs_boson/tables'
MODELS_DIR = f'{BASE_DIR}/results/higgs_boson/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Higgs Boson",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_kan.best_epoch,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/07_kan_sensitivity_k_2_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/07_kan_sensitivity_k_2_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/higgs_boson/tables/07_kan_sensitivity_k_2_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/higgs_boson/models/07_kan_sensitivity_k_2_69k.zip
